In [1]:
# Clone repo or pull latest updates
import os
if os.path.exists('fyp-dtcr-unet'):
    !cd fyp-dtcr-unet && git checkout main && git pull origin main
else:
    !git clone https://github.com/shadyOg/fyp-dtcr-unet.git

%cd fyp-dtcr-unet/dtcr-unet
!pip install -q -r requirements.txt


Cloning into 'fyp-dtcr-unet'...
remote: Enumerating objects: 233, done.
remote: Counting objects: 100% (233/233), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 233 (delta 128), reused 172 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (233/233), 2.47 MiB | 10.88 MiB/s, done.
Resolving deltas: 100% (128/128), done.
/kaggle/working/fyp-dtcr-unet/dtcr-unet


In [2]:
import os
from pathlib import Path

print("=== Scanning /kaggle/input ===")
input_root = Path('/kaggle/input')
for d in sorted(input_root.iterdir()):
    n_files = sum(1 for _ in d.rglob('*') if _.is_file())
    print(f"  📁 {d.name}/ ({n_files} files)")

# Auto-detect MosMedData path
MOSMED_PATH = None
for d in input_root.iterdir():
    nii_files = list(d.rglob('study_*.nii*'))
    if nii_files:
        MOSMED_PATH = str(d)
        print(f"\n✅ MosMedData found: {MOSMED_PATH} ({len(nii_files)} NIfTI files)")
        break
if not MOSMED_PATH:
    print("\n⚠️ MosMedData not found in /kaggle/input")

# Auto-detect LIDC-IDRI path
LIDC_PATH = None
for d in input_root.iterdir():
    lidc_dirs = list(d.rglob('LIDC-IDRI-*'))
    if lidc_dirs:
        LIDC_PATH = str(d)
        print(f"✅ LIDC-IDRI found: {LIDC_PATH} ({len(lidc_dirs)} patient/batch folders)")
        break
if not LIDC_PATH:
    for d in input_root.iterdir():
        if list(d.rglob('seg_LIDC-IDRI')):
            LIDC_PATH = str(d)
            print(f"✅ LIDC-IDRI (pre-segmented) found: {LIDC_PATH}")
            break
if not LIDC_PATH:
    print("⚠️ LIDC-IDRI not found in /kaggle/input")

PROCESSED_OUT = "/kaggle/working/data/processed"
print(f"\nTarget output folder: {PROCESSED_OUT}")


=== Scanning /kaggle/input ===
  📁 datasets/ (135328 files)

✅ MosMedData found: /kaggle/input/datasets (1160 NIfTI files)
✅ LIDC-IDRI found: /kaggle/input/datasets (601 patient/batch folders)

Target output folder: /kaggle/working/data/processed


In [3]:
# Build command with all available datasets
cmd = f'python data/preprocessing.py --out {PROCESSED_OUT} --size 256 --context 1 --labeled-ratio 0.4 --max-patients 100'

if MOSMED_PATH:
    cmd += f' --mosmed "{MOSMED_PATH}"'
if LIDC_PATH:
    cmd += f' --lidc "{LIDC_PATH}"'

print(f"Executing: {cmd}\n")
!{cmd}


Executing: python data/preprocessing.py --out /kaggle/working/data/processed --size 256 --context 1 --labeled-ratio 0.4 --max-patients 100 --mosmed "/kaggle/input/datasets" --lidc "/kaggle/input/datasets"

Found 50 paired MosMed volumes.
Total annotated studies with lesions: 50
Split breakdown: Train=30 studies, Val=10 studies, Test=10 studies.
Processing test split: 100%|████████████████████| 10/10 [00:14<00:00,  1.45s/it]

Preprocessing complete!
{
  "dataset": "MosMedData",
  "target_size": [
    256,
    256
  ],
  "total_extracted_slices": 1007,
  "splits": {
    "train": 640,
    "val": 181,
    "test": 186
  },
  "semi_supervised_breakdown": {
    "train_labeled": 227,
    "train_unlabeled": 413
  },
  "labeled_ratio": 0.4,
  "seed": 42
}
LIDC-IDRI Dataset Discovery:
  Search root: /kaggle/input/datasets
  Resolved root: /kaggle/input/datasets/washingtongold/lidcidri30
  Batch directories: ['LIDC-IDRI-0001-0200', 'LIDC-IDRI-0201-0400', 'LIDC-IDRI-0401-0600']
  XML annotations: /

In [4]:
import os
for split in ["train", "val", "test"]:
    img_dir = f"{PROCESSED_OUT}/{split}/images"
    mask_dir = f"{PROCESSED_OUT}/{split}/masks"
    lsf_dir = f"{PROCESSED_OUT}/{split}/level_sets"
    n_imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_masks = len(os.listdir(mask_dir)) if os.path.exists(mask_dir) else 0
    n_lsfs = len(os.listdir(lsf_dir)) if os.path.exists(lsf_dir) else 0
    print(f"Split [{split:>5}]: {n_imgs:>4} images | {n_masks:>4} masks | {n_lsfs:>4} level-set maps")


Split [train]: 1411 images | 1411 masks | 1411 level-set maps
Split [  val]:  480 images |  480 masks |  480 level-set maps
Split [ test]:  444 images |  444 masks |  444 level-set maps


In [5]:
!python train.py \
    --data /kaggle/working/data/processed \
    --epochs 100 \
    --batch-size 8 \
    --grad-accum 2 \
    --lr 3e-4 \
    --labeled-ratio 0.4 \
    --checkpoints /kaggle/working/checkpoints \
    --outputs /kaggle/working/outputs


Using device: cuda | AMP Mixed Precision: True
Data ready: 1411 train samples, 480 val samples.

Starting DTCR-U-Net training for 100 epochs...

Epoch 1/100 [Train]:   0%|                              | 0/128 [00:02<?, ?it/s]
Traceback (most recent call last):
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/train.py", line 340, in <module>
    main()
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/train.py", line 336, in main
    train(cfg, grad_accum_steps=args.grad_accum, use_amp=not args.no_amp)
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/train.py", line 238, in train
    train_losses = train_one_epoch(
                   ^^^^^^^^^^^^^^^^
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/train.py", line 61, in train_one_epoch
    loss, loss_dict = criterion(
                      ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [6]:
!python evaluate.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pth \
    --data /kaggle/working/data/processed \
    --batch-size 8 \
    --save-vis 20 \
    --out /kaggle/working/evaluation_results


Loading checkpoint from: /kaggle/working/checkpoints/best_model.pth
Traceback (most recent call last):
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/evaluate.py", line 134, in <module>
    main()
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/evaluate.py", line 124, in main
    evaluate_checkpoint(
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/evaluate.py", line 38, in evaluate_checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=dev)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__
    super().__in

In [7]:
import glob
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

viz_files = sorted(glob.glob('/kaggle/working/evaluation_results/comparisons/*.png'))[:8]
if viz_files:
    plt.figure(figsize=(16, 4 * len(viz_files)))
    for i, f in enumerate(viz_files):
        plt.subplot(len(viz_files), 1, i + 1)
        plt.imshow(Image.open(f))
        plt.axis('off')
        plt.title(Path(f).stem, fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No comparison images found. Check /kaggle/working/evaluation_results/")


No comparison images found. Check /kaggle/working/evaluation_results/
